# Smoke Test 

Goal of notebook - quickly check that the entire pipeline for adapting Llama-2-7B to the Bashkir language works:
- Loading the model in 4-bit quantization (QLoRA)
- Tokenization of Bashkir text
- LoRA training (continuous training) for 100 steps
- Reduced loss
- (optional) text generation

## Imports & Configs

In [1]:
!pip install -U bitsandbytes>=0.46.1 -q

In [2]:
import torch
from tqdm.auto import tqdm
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import numpy as np
import time
from torch.optim import AdamW
from torch.utils.data import DataLoader

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {mem_gb:.1f} GB")

PyTorch version: 2.10.0+cu128
Transformers version: 5.0.0
CUDA available: True
GPU device: Tesla T4
GPU memory: 15.6 GB


In [3]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e278979 (e278979-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN_METU"))


In [5]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    transformers.set_seed(seed)

set_seed(42)

# fast test configs
SMOKE_BATCH_SIZE = 4          # to save memory
SMOKE_MAX_STEPS = 100
SMOKE_LOG_EVERY = 20          # print loss each 20 steps
SMOKE_MAX_LEN = 128           # max len o tokens
SMOKE_NUM_SAMPLES = 200       # first 200 samples

print(f"Test config: batch_size={SMOKE_BATCH_SIZE}, steps={SMOKE_MAX_STEPS}, max_len={SMOKE_MAX_LEN}")

Test config: batch_size=4, steps=100, max_len=128


## Tokenizer & LLama

- Use `BitsAndBytesConfig` for 4-bit quantization (QLoRA).
- Tokenizer: use `eos_token` as `pad_token`, since Llama does not have a special pad_token.
- The model is loaded with `device_map="auto"` to automatically fit on the GPU (if available).
- Disable `use_cache` for training (saves memory).

In [6]:
MODEL_NAME = "meta-llama/Llama-2-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                 # turn on
    bnb_4bit_use_double_quant=True,    # double for economi
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Load tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token   # for Llama: pad_token = eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    use_cache=False             # save mem
)

total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded. Num of params: {total_params:.2f}B")

Load tokenizer...


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Model loaded. Num of params: 3.50B


## LoRA

- `prepare_model_for_kbit_training` - prepares a 4-bit model for training (freezes individual layers, adds hooks).
- `LoraConfig`:
  - `r=8` — LoRA rank (the higher, the more trainable parameters).
  - `lora_alpha=32` — scaling factor (usually alpha/r = 4).
  - `target_modules=["q_proj","v_proj"]` — apply LoRA to query and value projections in attention.
  - `lora_dropout=0.1` — dropout for regularization.
- We display the number of trainable parameters (should be ~4-8 million, <1% of 7B).

In [7]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Overall params: {total_params / 1e6:.2f}M")
print(f"trainable (LoRA): {trainable_params / 1e6:.2f}M ({100 * trainable_params / total_params:.2f}%)")

Overall params: 3504.61M
trainable (LoRA): 4.19M (0.12%)


## Dataset

- Load the `metuKKhud/bashqort-raw` dataset from Hugging Face.
- Let's take the first `SMOKE_NUM_SAMPLES` examples.
- Tokenize: trim or pad to `SMOKE_MAX_LEN`.
- Remove the original columns, leaving `input_ids` and `attention_mask`.
- Create a `DataLoader` for batches.

In [8]:
DATASET_PATH = "metuKKhud/bashqort-raw"
print(f"Load dataset {DATASET_PATH}...")
dataset = load_dataset(DATASET_PATH, split="texts_bashdram")

small_dataset = dataset.select(range(min(SMOKE_NUM_SAMPLES, len(dataset))))
print(f"Use {len(small_dataset)} rows.")


Load dataset metuKKhud/bashqort-raw...


README.md: 0.00B [00:00, ?B/s]

data/bash_news_articles-00000-of-00001.p(…):   0%|          | 0.00/38.5M [00:00<?, ?B/s]

data/bashgazet_articles-00000-of-00001.p(…):   0%|          | 0.00/1.46M [00:00<?, ?B/s]

data/neftcity_articles-00000-of-00001.pa(…):   0%|          | 0.00/451k [00:00<?, ?B/s]

...

Generating texts_kiskeufa split:   0%|          | 0/45 [00:00<?, ? examples/s]

Generating texts_kulturarb split:   0%|          | 0/1557 [00:00<?, ? examples/s]

Generating texts_president_rb split:   0%|          | 0/1879 [00:00<?, ? examples/s]

Generating texts_tabin split:   0%|          | 0/539 [00:00<?, ? examples/s]

Use 200 rows.


In [9]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=SMOKE_MAX_LEN,
        padding="max_length",
        return_tensors="pt"
    )

print("Tokenization...")
tokenized_dataset = small_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=small_dataset.column_names
)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])


Tokenization...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [10]:
dataloader = DataLoader(
    tokenized_dataset,
    batch_size=SMOKE_BATCH_SIZE,
    shuffle=True,
    drop_last=True
)
print(f"DataLoader with {len(dataloader)} batches (batch_size={SMOKE_BATCH_SIZE}) created.")

DataLoader with 50 batches (batch_size=4) created.


## Training!

- We use the AdamW optimizer with learning rate = 3e-4 (as in LlamaTurk for continual training).
- At each iteration: forward, loss calculation, backward, optimizer step.
- For causal LM labels = input_ids (the model learns to predict the next token).
- We log loss every 20 steps.
- We expect that the loss will decrease (for example, from ~8 to ~5 in 100 steps).

In [11]:
wandb.init(
    project="bashllama-smoke-test",
    name=f"smoke_test_{SMOKE_MAX_STEPS}steps",
    config={
        "model": "Llama-2-7b-hf",
        "quantization": "4bit_nf4",
        "lora_r": 8,
        "lora_alpha": 32,
        "batch_size": SMOKE_BATCH_SIZE,
        "max_steps": SMOKE_MAX_STEPS,
        "max_seq_len": SMOKE_MAX_LEN,
        "learning_rate": 3e-4,
        "optimizer": "AdamW"
    }
)

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260520_193623-wcascmb3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run smoke_test_100steps
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-smoke-test
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-smoke-test/runs/wcascmb3


In [12]:
optimizer = AdamW(model.parameters(), lr=3e-4)
model.train()

limited_dataloader = []
for i, batch in enumerate(dataloader):
    if i >= SMOKE_MAX_STEPS:
        break
    limited_dataloader.append(batch)

progress_bar = tqdm(limited_dataloader, desc="Training steps", unit="step")
losses = []
start_time = time.time()

for step, batch in enumerate(progress_bar):
    input_ids = batch["input_ids"].to(model.device)
    attention_mask = batch["attention_mask"].to(model.device)
    labels = input_ids.clone()
    
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )
    loss = outputs.loss
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    loss_val = loss.item()
    losses.append(loss_val)
    
    progress_bar.set_postfix({"loss": f"{loss_val:.4f}", "avg_loss": f"{sum(losses[-20:])/min(len(losses),20):.4f}"})
    
    wandb.log({
        "step": step + 1,
        "loss": loss_val,
        "avg_loss_20": sum(losses[-20:]) / min(len(losses), 20)
    })
    

end_time = time.time()
progress_bar.close()

print("-" * 50)
print(f"Training is done by {end_time - start_time:.1f} s.")
print(f"Init loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")


Training steps:   0%|          | 0/50 [00:00<?, ?step/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

--------------------------------------------------
Training is done by 516.1 s.
Init loss: 3.4239
Final loss: 2.5790


## Check

In [13]:
model.eval()
test_samples = [0, 50, 100, 150]
generation_table = wandb.Table(columns=["input_prefix", "generated_text"])

for idx in test_samples:
    if idx >= len(dataset):
        continue
    test_text = dataset[idx]["text"]
    prompt = test_text[:100]
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    generation_table.add_data(prompt, generated)
    print(f"--- Sample {idx} ---")
    print(f"Prompt: {prompt}...")
    print(f"Generated: {generated}\n")

wandb.log({"generations": generation_table})

--- Sample 0 ---
Prompt: Бөгөн - 4 декабрь -Мәжит Ғафури исемендәге Башҡорт дәүләт академия драма театрына -95 йыл! Барығыҙға...
Generated: Бөгөн - 4 декабрь -Мәжит Ғафури исемендәге Башҡорт дәүләт академия драма театрына -95 йыл! Барығыҙға һәм һайында - 18:00 саат сәхнәғәт.Ъылдың - 22:00 сәхнәғә

--- Sample 50 ---
Prompt: Инсценировка авторҙары - А.Абушахманов, Ш.Ғилманова. Бүләкләнеүсе мәҙәниәт һәм сәнғәт әһелдәре араһы...
Generated: Инсценировка авторҙары - А.Абушахманов, Ш.Ғилманова. Бүләкләнеүсе мәҙәниәт һәм сәнғәт әһелдәре араһында, 3 ай бер, 2017 йылда 18-30 сәғәт күрәүҙәнең өсөн Өфө

--- Sample 100 ---
Prompt: Конференция сиктәрендә «түңәрәк өҫтәлдәр», мәҙәниәт усаҡтары менән танышыу, Рәсәй мәҙәниәт министрлы...
Generated: Конференция сиктәрендә «түңәрәк өҫтәлдәр», мәҙәниәт усаҡтары менән танышыу, Рәсәй мәҙәниәт министрлыҡ исемендәге әҙеҙәрен алалары, тамашалары, танышыусылары һөнөнән аһырғ

--- Sample 150 ---
Prompt: Яңы йыл каникулдарында ла илһамланып эшләгән театр коллектив

In [14]:

wandb.finish()

wandb: uploading artifact run-wcascmb3-generations; updating run metadata
wandb: uploading artifact run-wcascmb3-generations
wandb: uploading media/table/generations_50_5d3220f282c6e3fab5b2.table.json; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb: avg_loss_20 ▇██▇▆▅▅▅▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
wandb:        loss ▇██▅▄▆▁█▅▆▅▅▅▅▅▅▅▅▃▄▅▄▄▃▃▂▃▃▁▄▁▄▃▂▄▁▄▃▃▂
wandb:        step ▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
wandb: 
wandb: Run summary:
wandb: avg_loss_20 2.71639
wandb:        loss 2.57905
wandb:        step 50
wandb: 
wandb: 🚀 View run smoke_test_100steps at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-smoke-test/runs/wcascmb3
wandb: ⭐️ View project at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-smoke-test
wandb: Synced 5 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260520_193623-wcascmb3/logs
